# Control de Flujo en Rust — Parte 2



Este notebook extiende los conceptos anteriores (if/else, match, shadowing) con:

- Operadores lógicos: `&&`, `||`, `!`
- Bucles `loop` en profundidad
- Bucles `while` y `while let`
- `continue` y `break` con etiquetas
- `for` con iteradores


---
# 1 — Operadores Lógicos



Los operadores lógicos combinan condiciones booleanas.

| Operador | Nombre | Resultado |
|----------|--------|-----------|
| `&&`     | AND    | `true` solo si **ambas** son verdaderas |
| `\|\|`   | OR     | `true` si **al menos una** es verdadera |
| `!`      | NOT    | **invierte** el valor booleano |

Son **cortocircuitantes**: Rust deja de evaluar en cuanto conoce el resultado.

## 1.1 — AND (`&&`)

In [4]:
fn main() {
    let edad = 20;
    let tiene_entrada = true;

    // Ambas condiciones deben ser true
    if edad >= 18 && tiene_entrada {
        println!("Puede ingresar al evento ✅");
    } else {
        println!("No puede ingresar ❌");
    }
}
main()

Puede ingresar al evento ✅


()

## 1.2 — OR (`||`)

In [5]:
fn main() {
    let es_admin = false;
    let es_moderador = true;

    // Alcanza con que UNA sea true
    if es_admin || es_moderador {
        println!("Tiene permisos elevados ✅");
    } else {
        println!("Usuario normal");
    }
}
main()

Tiene permisos elevados ✅


()

## 1.3 — NOT (`!`)

In [6]:
fn main() {
    let conectado = false;

    if !conectado {
        println!("El usuario está desconectado");
    }

    // También se puede negar directamente una expresión
    let bloqueado = false;
    let puede_actuar = !bloqueado;
    println!("¿Puede actuar? {}", puede_actuar);
}
main()

El usuario está desconectado
¿Puede actuar? true


()

## 1.4 — Combinando operadores

In [7]:
fn main() {
    let temperatura = 22;
    let llueve = false;
    let es_fin_de_semana = true;

    // Usar paréntesis para claridad cuando se mezclan operadores
    if (temperatura > 18 && !llueve) && es_fin_de_semana {
        println!("Buen día para salir al parque 🌳");
    } else if temperatura > 18 && !llueve {
        println!("Buen clima pero es día de semana 💼");
    } else {
        println!("Mejor quedarse en casa 🏠");
    }
}
main()

Buen día para salir al parque 🌳


()

## 1.5 — Cortocircuito (short-circuit evaluation)

Rust **no evalúa la segunda condición** si con la primera ya conoce el resultado.

- `false && <lo que sea>` → siempre `false`, no evalúa el segundo
- `true || <lo que sea>` → siempre `true`, no evalúa el segundo

Esto es importante cuando la segunda condición tiene **efectos secundarios** o puede **fallar**.

In [8]:
fn es_par(n: i32) -> bool {
    println!("  → evaluando es_par({})", n);
    n % 2 == 0
}

fn main() {
    println!("Caso AND con false primero:");
    // false && ... → no llega a llamar es_par
    if false && es_par(4) {
        println!("no llega acá");
    }

    println!("Caso OR con true primero:");
    // true || ... → no llega a llamar es_par
    if true || es_par(4) {
        println!("entra, pero sin evaluar es_par");
    }

    println!("Caso AND con true primero:");
    // true && ... → SÍ evalúa es_par
    if true && es_par(4) {
        println!("entra, y sí evaluó es_par");
    }
}
main()

Caso AND con false primero:
Caso OR con true primero:
entra, pero sin evaluar es_par
Caso AND con true primero:
  → evaluando es_par(4)
entra, y sí evaluó es_par


()

---
# 2 — El bucle `loop`



`loop` es el bucle más primitivo de Rust: **repite infinitamente** hasta un `break` explícito.

A diferencia de otros lenguajes, en Rust `loop` es una **expresión** que puede devolver un valor.

## 2.1 — loop básico con break

In [9]:
fn main() {
    let mut intentos = 0;

    loop {
        intentos += 1;
        println!("Intento #{}", intentos);

        if intentos == 3 {
            println!("Conexión lograda!");
            break; // sale del loop
        }
    }
}
main()

Intento #1
Intento #2
Intento #3
Conexión lograda!


()

## 2.2 — loop que devuelve un valor

La característica más especial de `loop` en Rust: `break valor` retorna un resultado.
Esto permite usar `loop` como una expresión en una asignación.

In [10]:
fn main() {
    let mut contador = 0;

    // El resultado del loop se asigna a `resultado`
    let resultado = loop {
        contador += 1;

        if contador == 5 {
            break contador * 2; // este valor "sale" del loop
        }
    };
    //          ↑ notar el punto y coma: loop es una expresión

    println!("contador llegó a: {}", contador);  // 5
    println!("resultado del loop: {}", resultado); // 10
}
main()

contador llegó a: 5
resultado del loop: 10


()

## 2.3 — continue: saltar una iteración

In [11]:
fn main() {
    let mut n = 0;

    loop {
        n += 1;

        if n > 10 { break; }

        // Saltar los números impares
        if n % 2 != 0 { continue; }

        println!("Par: {}", n);
    }
}
main()

Par: 2
Par: 4
Par: 6
Par: 8
Par: 10


()

## 2.4 — Etiquetas en loops anidados

Cuando tenés loops anidados, `break` y `continue` por defecto afectan al **loop más interno**.
Con etiquetas (`'nombre:`) podés controlar cuál loop romper.

In [12]:
fn main() {
    let mut fila = 0;

    'externo: loop {         // etiqueta para el loop externo
        fila += 1;
        let mut columna = 0;

        loop {               // loop interno (sin etiqueta)
            columna += 1;
            println!("fila={}, col={}", fila, columna);

            if columna == 3 {
                break;       // rompe solo el loop interno
            }
        }

        if fila == 2 {
            break 'externo;  // rompe el loop externo
        }
    }

    println!("Fin.");
}
main()

fila=1, col=1
fila=1, col=2
fila=1, col=3
fila=2, col=1
fila=2, col=2
fila=2, col=3
Fin.


()

---
# 3 — El bucle `while`



`while` evalúa una condición **antes de cada iteración**.
Si la condición es `false` desde el inicio, el cuerpo **nunca se ejecuta**.

```
┌─────────────────────────────────┐
│  while condición {              │
│      ┌─ evalúa condición        │
│      │  ↓ false → sale          │
│      │  ↓ true  → ejecuta body  │
│      └──────────────────────────│
│  }                              │
└─────────────────────────────────┘
```

## 3.1 — while básico

In [13]:
fn main() {
    let mut n = 1;

    while n <= 5 {
        println!("n = {}", n);
        n += 1; // sin esto sería un loop infinito!
    }

    println!("Terminó. n = {}", n); // n vale 6 acá
}
main()

n = 1
n = 2
n = 3
n = 4
n = 5
Terminó. n = 6


()

## 3.2 — while vs loop: ¿cuándo usar cada uno?

| Situación | Usar |
|-----------|------|
| Condición conocida de antemano | `while` |
| Necesitás retornar un valor del bucle | `loop` |
| La condición de salida está en el medio | `loop` + `break` |
| Iterar N veces sobre un rango | `for` |
| Bucle infinito (servidor, juego) | `loop` |

In [14]:
fn main() {
    // Con while: la condición está al inicio
    let mut pila = vec![3, 2, 1];
    while !pila.is_empty() {
        let valor = pila.pop().unwrap();
        println!("Procesando: {}", valor);
    }
    println!("Pila vacía.");
}
main()

Procesando: 1
Procesando: 2
Procesando: 3
Pila vacía.


()

## 3.3 — `while let`: desestructurando en la condición

`while let` combina `while` con pattern matching.
Ejecuta el cuerpo **mientras el patrón siga siendo verdadero**.
Es más idiomático que hacer `while Option.is_some()`.

In [15]:
fn main() {
    let mut stack = vec![10, 20, 30];

    // Mientras pop() devuelva Some(valor), lo desestructura directamente
    while let Some(tope) = stack.pop() {
        println!("Sacando de la pila: {}", tope);
    }
    // Cuando pop() devuelve None, el while termina
    println!("Pila vacía ✅");
}
main()

Sacando de la pila: 30
Sacando de la pila: 20
Sacando de la pila: 10
Pila vacía ✅


()

---
# 4 — `for` con iteradores



`for` en Rust no es solo para rangos numéricos — funciona con **cualquier iterador**.
Es la forma más segura e idiomática de recorrer colecciones.

## 4.1 — Iterar sobre un vector

In [16]:
fn main() {
    let frutas = vec!["manzana", "naranja", "pera"];

    for fruta in &frutas {  // & para no tomar ownership
        println!("Fruta: {}", fruta);
    }

    // El vector sigue disponible porque usamos &
    println!("Total de frutas: {}", frutas.len());
}
main()

Fruta: manzana
Fruta: naranja
Fruta: pera
Total de frutas: 3


()

## 4.2 — enumerate(): índice + valor

In [17]:
fn main() {
    let colores = vec!["rojo", "verde", "azul"];

    for (i, color) in colores.iter().enumerate() {
        println!("[{}] {}", i, color);
    }
}
main()

[0] rojo
[1] verde
[2] azul


()

## 4.3 — Rango al revés con .rev()

In [18]:
fn main() {
    print!("Cuenta regresiva: ");
    for n in (1..=5).rev() {
        print!("{} ", n);
    }
    println!("🚀");
}
main()

Cuenta regresiva: 5 4 3 2 1 🚀


()

## 4.4 — `break` y `continue` en bucles `for` y `while`

Para completar, aquí hay un par de ejemplos rápidos usando `break` y `continue` en bucles `for` y `while`.

In [ ]:
fn main() {
    println!("Ejemplo de continue en bucle for:");
    for i in 1..=5 {
        if i == 3 {
            println!("  -> Saltando el 3");
            continue; // Se salta esta iteración, no imprime y pasa al 4
        }
        println!("Número: {}", i);
    }
}
main()

In [ ]:
fn main() {
    println!("Ejemplo de break en bucle while:");
    let mut salud = 100;
    
    while salud > 0 {
        println!("Salud actual: {}", salud);
        salud -= 30;
        
        if salud <= 50 {
            println!("⚠️ Salud crítica! Cancelando bucle (Retirada)");
            break; // Rompe el bucle while incondicionalmente
        }
    }
}
main()

## 4.4 — `break` y `continue` en bucles `for` y `while`

Para completar, aquí hay un par de ejemplos rápidos usando `break` y `continue` en bucles `for` y `while`.

In [ ]:
fn main() {
    println!("Ejemplo de continue en bucle for:");
    for i in 1..=5 {
        if i == 3 {
            println!("  -> Saltando el 3");
            continue; // Se salta esta iteración, no imprime y pasa al 4
        }
        println!("Número: {}", i);
    }
}
main()

In [ ]:
fn main() {
    println!("Ejemplo de break en bucle while:");
    let mut salud = 100;
    
    while salud > 0 {
        println!("Salud actual: {}", salud);
        salud -= 30;
        
        if salud <= 50 {
            println!("⚠️ Salud crítica! Cancelando bucle (Retirada)");
            break; // Rompe el bucle while incondicionalmente
        }
    }
}
main()

---
# Resumen

| Concepto | Símbolo / Keyword | Cuándo usarlo |
|----------|-------------------|---------------|
| AND lógico | `&&` | Ambas condiciones deben ser true |
| OR lógico | `\|\|` | Al menos una debe ser true |
| NOT lógico | `!` | Invertir un booleano |
| Bucle infinito | `loop` | Cuando la salida es interna o necesitás retornar valor |
| Bucle condicional | `while` | Cuando la condición de salida está al principio |
| Pattern en bucle | `while let` | Cuando iterás sobre `Option` o `Result` |
| Bucle con iteradores | `for` | Recorrer rangos y colecciones |
| Saltar iteración | `continue` | Ignorar ciertos valores |
| Salir del bucle | `break` / `break valor` | Terminar, opcionalmente retornando algo |
| Loops anidados | `'etiqueta:` | Controlar cuál loop romper |
